In [1]:
import numpy as np
import torch as T
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
import os
import gymnasium as gym
from torch.distributions.normal import Normal
from gymnasium import wrappers


In [2]:


class ReplayBuffer():
    def __init__(self, max_size, input_shape, n_actions):
        self.mem_size = max_size
        self.mem_cntr = 0
        self.state_memory = np.zeros((self.mem_size, *input_shape))
        self.new_state_memory = np.zeros((self.mem_size, *input_shape))
        self.action_memory = np.zeros((self.mem_size, n_actions))
        self.reward_memory = np.zeros(self.mem_size)
        self.terminal_memory = np.zeros(self.mem_size)

    def store_transition(self, state, action, reward, state_, done):
        index = self.mem_cntr % self.mem_size

        self.state_memory[index] = state
        self.new_state_memory[index] = state_
        self.action_memory[index] = action
        self.reward_memory[index] = reward
        self.terminal_memory[index] = done

        self.mem_cntr += 1

    def sample_buffer(self, batch_size):
        max_mem = min(self.mem_cntr, self.mem_size)

        batch = np.random.choice(max_mem, batch_size)

        states = self.state_memory[batch]
        states_ = self.new_state_memory[batch]
        actions = self.action_memory[batch]
        rewards = self.reward_memory[batch]
        dones = self.terminal_memory[batch]

        return states, actions, rewards, states_, dones



In [3]:
class CriticNetwork(nn.Module):
  def __init__(self, beta, input_dim, n_actions, fc1_dim=256, fc2_dim=256):
    super(CriticNetwork,self).__init__()

    self.input_dims = input_dim
    self.fc1_dims = fc1_dim
    self.fc2_dims = fc2_dim
    self.n_actions = n_actions

    self.fc1 = nn.Linear(self.input_dims[0]+self.n_actions, self.fc1_dims)
    self.fc2 = nn.Linear(self.fc1_dims,self.fc2_dims)
    self.q = nn.Linear(self.fc2_dims,1)

    self.optimizer = optim.Adam(params=self.parameters(),lr=beta)

    self.device = T.device('cuda:0' if T.cuda.is_available() else 'cpu')

    self.to(self.device)

  def forward(self,state,action):

    combined = T.cat([state,action],dim=1)

    x = F.relu(self.fc1(combined))
    x = F.relu(self.fc2(x))
    q_out = self.q(x)
    return q_out

class ValueNetwork(nn.Module):
  def __init__(self, beta, input_dim, fc1_dim=256, fc2_dim=256):
    super(ValueNetwork,self).__init__()

    self.input_dims = input_dim
    self.fc1_dims = fc1_dim
    self.fc2_dims = fc2_dim

    self.fc1 = nn.Linear(self.input_dims[0], self.fc1_dims)
    self.fc2 = nn.Linear(self.fc1_dims, self.fc2_dims)
    self.q = nn.Linear(self.fc2_dims, 1)

    self.optimizer = optim.Adam(params=self.parameters(), lr=beta)

    self.device = T.device('cuda:0' if T.cuda.is_available() else 'cpu')

    self.to(self.device)

  def forward(self, state):

    state_value = F.relu(self.fc1(state))
    state_value = F.relu(self.fc2(state_value))
    q_out = self.q(state_value)
    return q_out


class ActorNetwork(nn.Module):
    def __init__(self,alhpa, input_dims, max_action, fc1_dim=256, fc2_dim=256,
                 n_actions=2):
        super(ActorNetwork,self).__init__()

        self.input_dims = input_dims
        self.fc1_dims = fc1_dim
        self.fc2_dims = fc2_dim
        self.n_actions = n_actions
        self.max_action = max_action
        #small noise that we added to std
        self.reparam_noise = 1e-6

        self.fc1 = nn.Linear(*self.input_dims, self.fc1_dims)
        self.fc2 = nn.Linear(self.fc1_dims, self.fc2_dims)

        #based on N_actions we have N normal distru
        self.mu = nn.Linear(self.fc2_dims, n_actions)
        self.sigma = nn.Linear(self.fc2_dims, n_actions)

        self.optimizer = optim.Adam(params=self.parameters(), lr=alhpa)
        self.device = T.device('cuda:0' if T.cuda.is_available() else 'cpu')
        self.to(self.device)

    def forward(self, state):
        prob = F.relu(self.fc1(state))
        prob = F.relu(self.fc2(prob))
        mu = self.mu(prob)
        sigma = self.sigma(prob)

        sigma = T.clamp(sigma, self.reparam_noise, 1)

        return mu, sigma
    #the whole idea of finding mu and std is to find dist with reparameterization trick

    def sample_normal(self, state, reparameterize=True):
        mu, sigma = self.forward(state)

        probability = Normal(mu, sigma)

        #if we use .sample still we care about probability and based on that we get value.
        #but for torch is not trackable as gradient

        #based on n_actions we have action here
        if reparameterize:
            actions = probability.resample()

        else:
            actions = probability.sample()

        #normalize action
        action = T.tanh(actions)*T.tensor(self.max_action).to(self.device)

        #log p for actions
        log_probs = probability.log_prob(actions)

        #That line is just a correction of the log-probability using the Jacobian,
        #because we applied tanh() to the Gaussian sample.
        log_probs -= T.log(1-action.pow(2)+self.reparam_noise)
        log_probs = log_probs.sum(1, keepdim=True)

        # if we didnt apply normalization and no need for jacobian all stuff was simpler
        # but in gym env, we need max range for action and that why we did this method like that...
        return action, log_probs



In [4]:
class Agent():
    def __init__(self, alpha=0.0003, beta=0.0003, input_dims=[8],
            env=None, gamma=0.99, n_actions=2, max_size=1000000, tau=0.005,
            layer1_size=256, layer2_size=256, batch_size=256, reward_scale=2):
        self.gamma = gamma
        self.tau = tau
        self.memory = ReplayBuffer(max_size, input_dims, n_actions)
        self.batch_size = batch_size
        self.n_actions = n_actions
        self.actor = ActorNetwork(alpha, input_dims, max_action=env.action_space.high)
        self.critic1 = CriticNetwork(beta, input_dims,n_actions)
        self.critic2 = CriticNetwork(beta, input_dims,n_actions)
        self.value = ValueNetwork(beta, input_dims)
        self.target_value = ValueNetwork(beta, input_dims)

        self.scale = reward_scale
        self.update_network_parameters(tau=1)

    def choose_action(self, observation):
        state = T.tensor(observation,dtype=T.float32).to(self.actor.device)
        if state.ndim == 1:
            state = state.unsqueeze(0)
        print(state.ndim)
        actions, _ = self.actor.sample_normal(state,reparameterize=False)
        return actions.cpu().detach().numpy()[0]

    def remember(self, state, action, reward, new_state, done):
        self.memory.store_transition(state, action, reward, new_state, done)

    def update_network_parameters(self, tau=None):
        if tau is None:
            tau = self.tau

        value_params = self.value.state_dict()
        target_value_params = self.target_value.state_dict()

        for name in value_params:
            target_value_params[name] = tau * value_params[name] + (1 - tau) * target_value_params[name]

        self.target_value.load_state_dict(target_value_params)


    def learn(self):
        if self.memory.mem_cntr < self.batch_size:
            return

        state, action, reward, new_state, done = \
                self.memory.sample_buffer(self.batch_size)

        reward = T.tensor(reward, dtype=T.float).to(self.actor.device)
        done = T.tensor(done).to(self.actor.device)
        state_ = T.tensor(new_state, dtype=T.float).to(self.actor.device)
        state = T.tensor(state, dtype=T.float).to(self.actor.device)
        action = T.tensor(action, dtype=T.float).to(self.actor.device)

        # instead of using
        value = self.value(state).view(-1)
        value_ = self.target_value(state_).view(-1)
        value_[done] = 0.0

        # learning critic, still we dont need to learn actor
        #optimize value
        actions,log_probs = self.actor.sample_normal(state,reparameterize=False)
        q_policy_1 = self.critic1(state,actions)
        q_policy_2 = self.critic2(state,actions)

        critic_value = T.min(q_policy_1,q_policy_2)
        critic_value = critic_value.view(-1) #Q

        self.value.optimizer.zero_grad()
        value_target = critic_value - log_probs

        value_loss = 0.5 * F.mse_loss(value, value_target)
        value_loss.backward(retain_graph=True)
        self.value.optimizer.step()


        #optimize actor
        actions, log_probs = self.actor.sample_normal(state,reparameterize=True)
        q1_new_policy = self.critic_1.forward(state, actions)
        q2_new_policy = self.critic_2.forward(state, actions)

        critic_value = T.min(q1_new_policy, q2_new_policy)
        critic_value = critic_value.view(-1)

        actor_loss = log_probs - critic_value
        actor_loss = T.mean(actor_loss).mean()

        self.actor.optimizer.zero_grad()
        actor_loss.backward(retain_graph=True)
        self.optimizer.step()


        #optimize critic
        self.critic_1.optimizer.zero_grad()
        self.critic_2.optimizer.zero_grad()
        q_hat = self.scale*reward + self.gamma*value_
        q1_old_policy = self.critic_1.forward(state, action).view(-1)
        q2_old_policy = self.critic_2.forward(state, action).view(-1)
        critic_1_loss = 0.5 * F.mse_loss(q1_old_policy, q_hat)
        critic_2_loss = 0.5 * F.mse_loss(q2_old_policy, q_hat)

        critic_loss = critic_1_loss + critic_2_loss
        critic_loss.backward()
        self.critic_1.optimizer.step()
        self.critic_2.optimizer.step()

        self.update_network_parameters()





In [ ]:
import gymnasium as gym

env = gym.make("InvertedPendulum-v5")
agent = Agent(input_dims=env.observation_space.shape, env=env,
            n_actions=env.action_space.shape[0])